# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Run the training script (SKIP THIS if you already have the model)
# !python train_hf.py

In [ ]:
# 4. FIND and Deploy Existing Model to Hugging Face Space
# This cell will SEARCH for your unzipped model anywhere in Colab.

import os
import shutil

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
SPACE_ID = SPACE_ID.strip()

# --- 1. SEARCH FOR MODEL ---
print("🔍 Searching for unzipped model directory 'trocr_finetuned_iam_hf'...")
found_model_path = None

# Search common locations
search_paths = [
    "/content",
    "/content/handwriting_recog",
    "/content/handwriting_recog/models",
    "."
]

for root_path in search_paths:
    if not os.path.exists(root_path):
        continue
        
    for dirpath, dirnames, filenames in os.walk(root_path):
        if "trocr_finetuned_iam_hf" in dirnames:
            full_path = os.path.join(dirpath, "trocr_finetuned_iam_hf")
            # Verify it contains model files
            if "config.json" in os.listdir(full_path) or "model.safetensors" in os.listdir(full_path):
                found_model_path = full_path
                print(f"✅ FOUND MODEL AT: {found_model_path}")
                break
    if found_model_path:
        break

if not found_model_path:
    print("❌ CRITICAL: Could not find 'trocr_finetuned_iam_hf' directory anywhere!")
    print("Since the runtime was reset, your unzipped files might have been deleted.")
    print("You may need to upload the zip file again or retrain.")
    # Stop here if not found
else:
    # --- 2. Reduce Size: Remove Checkpoints ---
    print("🧹 Cleaning up intermediate checkpoints to save space...")
    checkpoints = [d for d in os.listdir(found_model_path) if d.startswith('checkpoint-')]
    for ckpt in checkpoints:
        ckpt_path = os.path.join(found_model_path, ckpt)
        print(f"Removing {ckpt_path}...")
        shutil.rmtree(ckpt_path)

    # --- 3. Deploy to Spaces ---
    if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
        print("⚠️ Please enter your Hugging Face Write Token above!")
    else:
        print("🚀 Starting deployment to Hugging Face Space...")
        
        # Clean up any existing space_repo to prevent nesting issues
        if os.path.exists("space_repo"):
            print("🗑️ Removing existing space_repo directory...")
            shutil.rmtree("space_repo")
        
        # Install Git LFS
        !git lfs install
        
        # Configure Git
        !git config --global user.email "colab@example.com"
        !git config --global user.name "Colab User"
        
        # Clone your Hugging Face Space
        repo_url = f"https://{HF_TOKEN}@huggingface.co/spaces/{SPACE_ID}"
        !git clone {repo_url} space_repo
        
        # Check if app_gradio.py exists in CURRENT directory
        # If not, download it from repo
        if not os.path.exists("app_gradio.py"):
             print("⚠️ app_gradio.py not found locally. Downloading from repo...")
             !wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/app_gradio.py
             !wget https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/requirements.txt
             !mkdir -p utils
             !wget -P utils https://raw.githubusercontent.com/Bhuvan-018/handwriting_recog/main/utils/preprocessing.py

        # Copy model files to the Space repo
        print(f"📦 Moving model files from {found_model_path}...")
        !mkdir -p space_repo/models/trocr_finetuned_iam_hf
        
        # MOVE files instead of COPY to save disk space
        !mv {found_model_path}/* space_repo/models/trocr_finetuned_iam_hf/
        
        # Remove empty source dir
        if os.path.exists(found_model_path):
            shutil.rmtree(found_model_path)
            print("🧹 Removed source model directory.")
        
        # Copy app files (ensure they are up to date from the repo)
        print("📄 Copying app files...")
        !cp app_gradio.py space_repo/app.py
        !cp requirements.txt space_repo/
        if os.path.exists("utils"):
             !cp -r utils space_repo/
        
        # Commit and Push
        print("⬆️ Pushing to Hugging Face (this may take a few minutes)...")
        # Change directory to space_repo ONLY for git operations
        os.chdir("space_repo")
        
        # Set remote url again just to be safe
        !git remote set-url origin {repo_url}
        
        !git lfs track "*.bin"
        !git lfs track "*.safetensors"
        !git add .
        !git commit -m "Deploy fine-tuned model from Colab"
        
        # Use robust push
        !git push
        print("✅ Successfully deployed to Hugging Face Space!")
        # Go back to parent directory
        os.chdir("..")